# Model Evaluation — Performance Test Classification

This notebook runs the full training pipeline and provides detailed evaluation:
- Confusion matrices for all models
- ROC curves comparison
- Precision / Recall / F1 / Balanced Accuracy
- Feature importance analysis
- Test case predictions with explanations

**Prerequisite:** Either a live DB connection (`.env`) or a pre-exported CSV from `src.extract`.

In [ ]:
import sys
import os

# Ensure we're in the project root directory (same as predict_demo.ipynb)
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print(f"📂 Working directory: {os.getcwd()}")

sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import sqlalchemy

from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay,
)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
print('Imports ready')

ModuleNotFoundError: No module named 'sqlalchemy'

## 1. Run Training Pipeline

Execute the full extract → features → train → evaluate pipeline.

**Two options:**
- **Option A:** From live database (requires `.env` configuration)
- **Option B:** From pre-exported CSV (database not required!)

A pre-exported CSV is included at `../data_exports/training_data.csv` for offline use.

In [1]:
from src.train import run_pipeline

# Option A: From live database (requires .env)
# results, run_df, baselines = run_pipeline()

# Option B: From pre-exported CSV (NO DATABASE REQUIRED!)
results, run_df, baselines = run_pipeline(csv_path='../data_exports/training_data.csv')

print('✅ Training pipeline complete!')

ModuleNotFoundError: No module named 'src'

## 2. Metrics Comparison Table

In [2]:
metrics_rows = []
for name, res in results.items():
    m = res['metrics']
    metrics_rows.append({
        'Model': name,
        'Accuracy': f"{m['accuracy']:.4f}",
        'Balanced Acc': f"{m['balanced_accuracy']:.4f}",
        'Precision': f"{m['precision']:.4f}",
        'Recall': f"{m['recall']:.4f}",
        'F1-Score': f"{m['f1_score']:.4f}",
        'ROC-AUC': f"{m['roc_auc']:.4f}" if m.get('roc_auc') else 'N/A',
        'CV F1 (5-fold)': f"{m['cv_f1_mean']:.4f} ± {m['cv_f1_std']:.4f}",
    })

metrics_df = pd.DataFrame(metrics_rows).set_index('Model')
metrics_df.style.set_caption('Model Performance Comparison')

NameError: name 'results' is not defined

## 3. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4))
if len(results) == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, results.items()):
    m = res['metrics']
    cm = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Fail', 'Pass'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name}\nAcc={m["accuracy"]:.3f} | F1={m["f1_score"]:.3f}')

plt.tight_layout()
plt.show()

# Print TP/TN/FP/FN for each model
for name, res in results.items():
    m = res['metrics']
    print(f'{name}: TP={m["TP"]} TN={m["TN"]} FP={m["FP"]} FN={m["FN"]}')

## 4. ROC Curves

In [ ]:
# Reconstruct y_test from run_df using the same split
from sklearn.model_selection import train_test_split
from src.train import TEST_SIZE, RANDOM_STATE
from src.features import MODEL_FEATURES

runs_unique = run_df[['testplan', 'label_pass_fail']].drop_duplicates()
_, test_plans = train_test_split(
    runs_unique['testplan'], test_size=TEST_SIZE,
    random_state=RANDOM_STATE, stratify=runs_unique['label_pass_fail']
)
test_df = run_df[run_df['testplan'].isin(test_plans)]
y_test = test_df['label_pass_fail']

fig, ax = plt.subplots(figsize=(8, 6))
for name, res in results.items():
    if res['y_proba'] is not None and res['metrics'].get('roc_auc'):
        fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
        ax.plot(fpr, tpr, label=f"{name} (AUC={res['metrics']['roc_auc']:.3f})")

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Model Comparison')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Feature Importance

In [ ]:
for name in ['Random Forest', 'Decision Tree']:
    if name not in results:
        continue
    model = results[name]['model']
    if not hasattr(model, 'feature_importances_'):
        continue

    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(range(len(MODEL_FEATURES)), importances[indices[::-1]], color='steelblue')
    ax.set_yticks(range(len(MODEL_FEATURES)))
    ax.set_yticklabels([MODEL_FEATURES[i] for i in indices[::-1]])
    ax.set_xlabel('Importance')
    ax.set_title(f'Feature Importance — {name}')
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

    print(f'\n{name} — Top 5 features:')
    for i in range(min(5, len(MODEL_FEATURES))):
        print(f'  {MODEL_FEATURES[indices[i]]}: {importances[indices[i]]:.4f}')

## 6. Test Case Predictions

In [ ]:
import joblib
from src.predict import predict_and_explain, load_test_cases

model = joblib.load('../models/model.pkl')
scaler = joblib.load('../models/scaler.pkl')
cases = load_test_cases('../test_cases/test_cases.json')

predict_and_explain(model, scaler, cases)

## 7. Summary

### Results
- **Best model** and its metrics are documented in `models/metrics.json`
- **Confusion matrix** shows TP/TN/FP/FN breakdown
- **ROC-AUC** indicates discrimination ability between pass and fail
- **Feature importance** shows which performance dimensions drive the classification

### Deliverables
- `models/model.pkl` — trained classifier
- `models/scaler.pkl` — fitted feature scaler
- `models/baselines.pkl` — per-transaction baselines
- `test_cases/test_cases.json` — 3 representative test cases
- `src/predict.py` — standalone prediction script (run with `python -m src.predict`)